In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, OneHotEncoder
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.decomposition import PCA

df = pd.read_csv('heart (1).csv')

print("First 5 rows of the dataset:")
display(df.head())
print("\nInformation about the dataset:")
display(df.info())

First 5 rows of the dataset:


,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0



Information about the dataset:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


None

In [6]:
X = df.drop('HeartDisease', axis=1)
y = df['HeartDisease']

categorical_cols = X.select_dtypes(include=['object']).columns
numerical_cols = X.select_dtypes(include=np.number).columns

print("\nUnique values for original categorical columns:")
for col in categorical_cols:
    print(f"{col}: {X[col].unique()}")

X_encoded = X.copy()

X_encoded = pd.get_dummies(X_encoded, columns=categorical_cols, drop_first=True)

print("\nDataFrame after One-Hot Encoding:")
display(X_encoded.head())

original_numerical_cols = numerical_cols.tolist()

X_train, X_test, y_train, y_test = train_test_split(X_encoded, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train[original_numerical_cols] = scaler.fit_transform(X_train[original_numerical_cols])
X_test[original_numerical_cols] = scaler.transform(X_test[original_numerical_cols])

print("\nScaled training data (first 5 rows of relevant columns):")
display(X_train[original_numerical_cols].head())


Unique values for original categorical columns:
Sex: ['M' 'F']
ChestPainType: ['ATA' 'NAP' 'ASY' 'TA']
RestingECG: ['Normal' 'ST' 'LVH']
ExerciseAngina: ['N' 'Y']
ST_Slope: ['Up' 'Flat' 'Down']

DataFrame after One-Hot Encoding:


,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak,Sex_M,ChestPainType_ATA,ChestPainType_NAP,ChestPainType_TA,RestingECG_Normal,RestingECG_ST,ExerciseAngina_Y,ST_Slope_Flat,ST_Slope_Up
0,40,140,289,0,172,0.0,True,True,False,False,True,False,False,False,True
1,49,160,180,0,156,1.0,False,False,True,False,True,False,False,True,False
2,37,130,283,0,98,0.0,True,True,False,False,False,True,False,False,True
3,48,138,214,0,108,1.5,False,False,False,False,True,False,True,True,False
4,54,150,195,0,122,0.0,True,False,True,False,True,False,False,False,True



Scaled training data (first 5 rows of relevant columns):


,Age,RestingBP,Cholesterol,FastingBS,MaxHR,Oldpeak
795,-1.245067,-0.708985,0.372803,1.842609,2.284353,-0.097061
25,-1.886236,-0.166285,0.086146,-0.542709,1.652241,-0.836286
84,0.250993,0.919115,0.123134,1.842609,-0.441628,0.087745
10,-1.779375,-0.166285,0.104640,-0.542709,0.229991,-0.836286
344,-0.283314,-0.708985,-1.846478,1.842609,-1.271274,-0.836286


In [7]:
svm_model = SVC(random_state=42)
log_reg_model = LogisticRegression(random_state=42, max_iter=1000)
rf_model = RandomForestClassifier(random_state=42)

accuracy_before_pca = {}

print("\n--- Model Training and Evaluation (Before PCA) ---")

svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)
accuracy_before_pca['SVM'] = accuracy_score(y_test, y_pred_svm)
print(f"SVM Accuracy (Before PCA): {accuracy_before_pca['SVM']:.4f}")

log_reg_model.fit(X_train, y_train)
y_pred_log_reg = log_reg_model.predict(X_test)
accuracy_before_pca['Logistic Regression'] = accuracy_score(y_test, y_pred_log_reg)
print(f"Logistic Regression Accuracy (Before PCA): {accuracy_before_pca['Logistic Regression']:.4f}")

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)
accuracy_before_pca['Random Forest'] = accuracy_score(y_test, y_pred_rf)
print(f"Random Forest Accuracy (Before PCA): {accuracy_before_pca['Random Forest']:.4f}")


--- Model Training and Evaluation (Before PCA) ---
SVM Accuracy (Before PCA): 0.8587
Logistic Regression Accuracy (Before PCA): 0.8533
Random Forest Accuracy (Before PCA): 0.8750


In [8]:
pca = PCA(n_components=0.95, random_state=42)
pca.fit(X_train)

X_train_pca = pca.transform(X_train)
X_test_pca = pca.transform(X_test)

print(f"\nNumber of components selected by PCA: {pca.n_components_}")

accuracy_after_pca = {}

print("\n--- Model Training and Evaluation (After PCA) ---")

svm_model_pca = SVC(random_state=42)
svm_model_pca.fit(X_train_pca, y_train)
y_pred_svm_pca = svm_model_pca.predict(X_test_pca)
accuracy_after_pca['SVM'] = accuracy_score(y_test, y_pred_svm_pca)
print(f"SVM Accuracy (After PCA): {accuracy_after_pca['SVM']:.4f}")

log_reg_model_pca = LogisticRegression(random_state=42, max_iter=1000)
log_reg_model_pca.fit(X_train_pca, y_train)
y_pred_log_reg_pca = log_reg_model_pca.predict(X_test_pca)
accuracy_after_pca['Logistic Regression'] = accuracy_score(y_test, y_pred_log_reg_pca)
print(f"Logistic Regression Accuracy (After PCA): {accuracy_after_pca['Logistic Regression']:.4f}")

rf_model_pca = RandomForestClassifier(random_state=42)
rf_model_pca.fit(X_train_pca, y_train)
y_pred_rf_pca = rf_model_pca.predict(X_test_pca)
accuracy_after_pca['Random Forest'] = accuracy_score(y_test, y_pred_rf_pca)
print(f"Random Forest Accuracy (After PCA): {accuracy_after_pca['Random Forest']:.4f}")


Number of components selected by PCA: 10

--- Model Training and Evaluation (After PCA) ---
SVM Accuracy (After PCA): 0.8424
Logistic Regression Accuracy (After PCA): 0.8315
Random Forest Accuracy (After PCA): 0.8750
